# Cinema Revenue Prediction with Apache Spark

このNotebookでは、Apache Sparkを使用してCinemaデータセットを探索し、興行収入予測の回帰モデルを構築します。

In [1]:
// Spark 依存関係の読み込み（Scala 2.13 を明示的に指定）
import $ivy.`org.apache.spark:spark-sql_2.13:3.5.0`
import $ivy.`org.apache.spark:spark-mllib_2.13:3.5.0`

println("Spark 依存関係が正常にロードされました")

Spark 依存関係が正常にロードされました


import $ivy.$
import $ivy.$

## 1. 環境設定とライブラリのインポート

In [2]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.ml.{Pipeline, PipelineModel}
import org.apache.spark.ml.regression.LinearRegression
import org.apache.spark.ml.feature.{StringIndexer, OneHotEncoder, VectorAssembler}
import org.apache.spark.ml.evaluation.RegressionEvaluator

// SparkSessionの作成
val spark = SparkSession.builder()
  .appName("CinemaExploration")
  .master("local[*]")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

println("Spark Session created successfully!")
println(s"Spark version: ${spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/04 12:56:23 INFO SparkContext: Running Spark version 3.5.0
25/11/04 12:56:23 INFO SparkContext: OS info Windows 11, 10.0, amd64
25/11/04 12:56:23 INFO SparkContext: Java version 21.0.2
25/11/04 12:56:23 WARN Shell: Did not find winutils.exe: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://wiki.apache.org/hadoop/WindowsProblems
25/11/04 12:56:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/04 12:56:23 INFO ResourceUtils: ==============================================================
25/11/04 12:56:23 INFO ResourceUtils: No custom resources configured for spark.driver.
25/11/04 12:56:23 INFO ResourceUtils: ==============================================================
25/11/04 12:56:23 INFO SparkContext: Submitted application: CinemaExploration
25/1

Spark Session created successfully!
Spark version: 3.5.0


import org.apache.spark.sql.SparkSession
import org.apache.spark.ml.{Pipeline, PipelineModel}
import org.apache.spark.ml.regression.LinearRegression
import org.apache.spark.ml.feature.{StringIndexer, OneHotEncoder, VectorAssembler}
import org.apache.spark.ml.evaluation.RegressionEvaluator
spark: SparkSession = org.apache.spark.sql.SparkSession@32bbaf60

## 2. データの読み込み

In [3]:
// データの読み込み
val df = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("../data/cinema.csv")

println(s"データ件数: ${df.count()}")
println("\nスキーマ:")
df.printSchema()

データ件数: 500

スキーマ:
root
 |-- budget: integer (nullable = true)
 |-- popularity: double (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- vote_average: double (nullable = true)
 |-- genre: string (nullable = true)
 |-- revenue: integer (nullable = true)



df: org.apache.spark.sql.package.DataFrame = [budget: int, popularity: double ... 4 more fields]

## 3. データの概要確認

In [4]:
// 最初の5行を表示
df.show(5, truncate = false)

+------+----------+-------+------------+------+-------+
|budget|popularity|runtime|vote_average|genre |revenue|
+------+----------+-------+------------+------+-------+
|42905 |11.13     |154    |3.5         |Comedy|43797  |
|10144 |73.65     |146    |7.7         |Action|20277  |
|39698 |42.19     |63     |1.8         |Comedy|40885  |
|34118 |60.2      |131    |2.8         |Horror|27300  |
|15446 |44.92     |95     |8.3         |Action|23882  |
+------+----------+-------+------------+------+-------+
only showing top 5 rows



In [5]:
// 統計情報
df.describe("budget", "popularity", "runtime", "vote_average", "revenue").show()

+-------+----------------+-----------------+-----------------+-----------------+------------------+
|summary|          budget|       popularity|          runtime|     vote_average|           revenue|
+-------+----------------+-----------------+-----------------+-----------------+------------------+
|  count|             500|              500|              500|              500|               500|
|   mean|        25098.92|49.61746000000003|          121.524|5.505400000000002|         25784.872|
| stddev|14047.7039128837|29.08934121793387|33.13132646887495|2.549633719617078|14550.996434130151|
|    min|            1212|             0.16|               60|              1.0|              2146|
|    max|           49926|            99.93|              180|             10.0|             80377|
+-------+----------------+-----------------+-----------------+-----------------+------------------+



In [5]:
// ジャンルごとの件数
df.groupBy("genre").count().orderBy($"count".desc).show()

cmd6.sc:2: value $ is not a member of StringContext
df.groupBy("genre").count().orderBy($"count".desc).show()
                                    ^
Compilation Failed

## 4. ジャンルのOneHotエンコーディング

In [6]:
// ステップ1: StringIndexer でジャンルを数値に変換
val indexer = new StringIndexer()
  .setInputCol("genre")
  .setOutputCol("genre_index")

// ステップ2: OneHotEncoder でダミー変数化
val encoder = new OneHotEncoder()
  .setInputCol("genre_index")
  .setOutputCol("genre_vec")
  .setDropLast(false)

val encodePipeline = new Pipeline().setStages(Array(indexer, encoder))
val encodedDf = encodePipeline.fit(df).transform(df)

println("ジャンルのエンコーディング完了")
encodedDf.select("genre", "genre_index", "genre_vec").show(5, truncate = false)

ジャンルのエンコーディング完了
+------+-----------+-------------+
|genre |genre_index|genre_vec    |
+------+-----------+-------------+
|Comedy|3.0        |(4,[3],[1.0])|
|Action|1.0        |(4,[1],[1.0])|
|Comedy|3.0        |(4,[3],[1.0])|
|Horror|0.0        |(4,[0],[1.0])|
|Action|1.0        |(4,[1],[1.0])|
+------+-----------+-------------+
only showing top 5 rows



indexer: StringIndexer = strIdx_19e323b6a441
encoder: OneHotEncoder = oneHotEncoder_329480069120
encodePipeline: Pipeline = pipeline_1ad3b816acec
encodedDf: org.apache.spark.sql.package.DataFrame = [budget: int, popularity: double ... 6 more fields]

## 5. 特徴量の統合

In [7]:
// 全ての特徴量を1つのベクトルに統合
val assembler = new VectorAssembler()
  .setInputCols(Array(
    "budget",
    "popularity",
    "runtime",
    "vote_average",
    "genre_vec"
  ))
  .setOutputCol("features")
  .setHandleInvalid("skip")

val assembledDf = assembler.transform(encodedDf)

println(s"準備後のデータ件数: ${assembledDf.count()}")
assembledDf.select("features", "revenue").show(5, truncate = false)

準備後のデータ件数: 500
+-----------------------------------------+-------+
|features                                 |revenue|
+-----------------------------------------+-------+
|[42905.0,11.13,154.0,3.5,0.0,0.0,0.0,1.0]|43797  |
|[10144.0,73.65,146.0,7.7,0.0,1.0,0.0,0.0]|20277  |
|[39698.0,42.19,63.0,1.8,0.0,0.0,0.0,1.0] |40885  |
|[34118.0,60.2,131.0,2.8,1.0,0.0,0.0,0.0] |27300  |
|[15446.0,44.92,95.0,8.3,0.0,1.0,0.0,0.0] |23882  |
+-----------------------------------------+-------+
only showing top 5 rows



assembler: VectorAssembler = VectorAssembler: uid=vecAssembler_1a4d1eeb6727, handleInvalid=skip, numInputCols=5
assembledDf: org.apache.spark.sql.package.DataFrame = [budget: int, popularity: double ... 7 more fields]

## 6. データの分割

In [8]:
// 訓練データとテストデータに分割
val Array(trainData, testData) = assembledDf.randomSplit(Array(0.7, 0.3), seed = 42)

println(s"訓練データ: ${trainData.count()} 件")
println(s"テストデータ: ${testData.count()} 件")

訓練データ: 374 件
テストデータ: 126 件


trainData: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [budget: int, popularity: double ... 7 more fields]
testData: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [budget: int, popularity: double ... 7 more fields]

## 7. Linear Regressionモデルの訓練

In [9]:
// Linear Regressionモデルの作成
val lr = new LinearRegression()
  .setLabelCol("revenue")
  .setFeaturesCol("features")
  .setMaxIter(100)
  .setRegParam(0.1)
  .setElasticNetParam(0.0)

val pipeline = new Pipeline().setStages(Array(lr))

// モデルの訓練
println("モデルを訓練中...")
val model = pipeline.fit(trainData)
println("訓練完了！")

モデルを訓練中...
訓練完了！


lr: LinearRegression = linReg_4b706a854a84
pipeline: Pipeline = pipeline_d179d82126fe
model: PipelineModel = pipeline_d179d82126fe

## 8. モデルの評価

In [10]:
// テストデータで予測
val predictions = model.transform(testData)

// 評価メトリクスの計算
val evaluator = new RegressionEvaluator()
  .setLabelCol("revenue")
  .setPredictionCol("prediction")

val r2 = evaluator.setMetricName("r2").evaluate(predictions)
val rmse = evaluator.setMetricName("rmse").evaluate(predictions)
val mae = evaluator.setMetricName("mae").evaluate(predictions)

println(f"R² Score: ${r2 * 100}%.2f%%")
println(f"RMSE: $rmse%.2f")
println(f"MAE: $mae%.2f")

R² Score: 91.62%
RMSE: 4442.93
MAE: 3465.43


predictions: org.apache.spark.sql.package.DataFrame = [budget: int, popularity: double ... 8 more fields]
evaluator: RegressionEvaluator = RegressionEvaluator: uid=regEval_983aef5ce39b, metricName=mae, throughOrigin=false
r2: Double = 0.9162425313861551
rmse: Double = 4442.929199743235
mae: Double = 3465.427423515989

## 9. 予測結果の確認

In [11]:
// 予測結果のサンプル表示
predictions.select(
  "budget", "popularity", "runtime", "vote_average", "genre",
  "revenue", "prediction"
).show(10, truncate = false)

+------+----------+-------+------------+------+-------+------------------+
|budget|popularity|runtime|vote_average|genre |revenue|prediction        |
+------+----------+-------+------------+------+-------+------------------+
|1418  |41.24     |167    |6.7         |Action|6592   |13908.228844052954|
|2005  |72.24     |112    |8.8         |Comedy|8956   |8886.242663343335 |
|2378  |58.83     |89     |6.3         |Action|8398   |14547.016002185861|
|2437  |97.79     |144    |4.6         |Action|11384  |16623.477196042026|
|3040  |14.28     |137    |7.1         |Horror|3726   |-2471.322320633196|
|3103  |80.5      |111    |3.4         |Comedy|6382   |8839.743071638737 |
|3280  |12.62     |90     |8.0         |Drama |4278   |1874.1486054044701|
|3968  |42.37     |149    |1.6         |Action|5625   |14714.051951259826|
|4132  |86.58     |102    |3.5         |Drama |6675   |5040.873717315227 |
|4314  |78.68     |91     |9.9         |Comedy|9422   |11300.204930590278|
+------+----------+------

In [12]:
// 実際の値と予測値の比較（誤差を計算）
import org.apache.spark.sql.functions._

val comparison = predictions.select(
  col("revenue").as("actual"),
  col("prediction"),
  abs(col("revenue") - col("prediction")).as("error"),
  (abs(col("revenue") - col("prediction")) / col("revenue") * 100).as("error_pct")
)

comparison.describe("actual", "prediction", "error", "error_pct").show()

+-------+------------------+------------------+------------------+-------------------+
|summary|            actual|        prediction|             error|          error_pct|
+-------+------------------+------------------+------------------+-------------------+
|  count|               126|               126|               126|                126|
|   mean| 24717.23015873016| 25239.16799345566| 3465.427423515989| 22.546026603949734|
| stddev|15413.025148445167|14126.243155783332| 2791.464867671558|  29.48635921188773|
|    min|              3726|-2471.322320633196| 48.63734263972492|0.40143926202134167|
|    max|             70953| 57808.90747634965|15817.487249697115|  166.3264176230058|
+-------+------------------+------------------+------------------+-------------------+



import org.apache.spark.sql.functions._
comparison: org.apache.spark.sql.package.DataFrame = [actual: int, prediction: double ... 2 more fields]

## 10. モデルの係数確認

In [13]:
// Linear Regressionモデルの係数を表示
val lrModel = model.stages(0).asInstanceOf[org.apache.spark.ml.regression.LinearRegressionModel]

println("モデルの係数:")
println(s"Intercept: ${lrModel.intercept}")
println(s"Coefficients: ${lrModel.coefficients}")
println()
println("訓練セットでの性能:")
println(s"RMSE: ${lrModel.summary.rootMeanSquaredError}")
println(s"R²: ${lrModel.summary.r2}")

モデルの係数:
Intercept: -1981.0909322363848
Coefficients: [0.9013996805540295,46.33557299438677,12.01770132358505,260.54640232647625,-7388.462937490038,8947.638982349119,-2852.0706834999874,2073.9545542398737]

訓練セットでの性能:
RMSE: 4177.108397550268
R²: 0.913872088847059


lrModel: org.apache.spark.ml.regression.LinearRegressionModel = LinearRegressionModel: uid=linReg_4b706a854a84, numFeatures=8

## 11. 特徴量の重要度確認

In [14]:
// 係数の絶対値を特徴量の重要度として表示
val featureNames = Array("budget", "popularity", "runtime", "vote_average") ++ 
                   (0 until lrModel.coefficients.size - 4).map(i => s"genre_$i")

val coefficients = lrModel.coefficients.toArray

println("特徴量の重要度（係数の絶対値）:")
featureNames.zip(coefficients).sortBy(-_._2.abs).take(10).foreach { case (name, coef) =>
  println(f"$name%-20s: $coef%.4f")
}

特徴量の重要度（係数の絶対値）:
genre_1             : 8947.6390
genre_0             : -7388.4629
genre_2             : -2852.0707
genre_3             : 2073.9546
vote_average        : 260.5464
popularity          : 46.3356
runtime             : 12.0177
budget              : 0.9014


featureNames: Array[String] = Array(
  "budget",
  "popularity",
  "runtime",
  "vote_average",
  "genre_0",
  "genre_1",
  "genre_2",
  "genre_3"
)
coefficients: Array[Double] = Array(
  0.9013996805540295,
  46.33557299438677,
  12.01770132358505,
  260.54640232647625,
  -7388.462937490038,
  8947.638982349119,
  -2852.0706834999874,
  2073.9545542398737
)

## 12. クリーンアップ

In [15]:
// SparkSessionの停止
// spark.stop()
println("完了！")

完了！
